# Data Analytics and Visualization in Data-Centric Engineering

This workbook supports the presentation for the workshop. Use it as the working copy while the slides introduce the main ideas.

The example uses three months of 10-minute wind turbine telemetry to show cleaning, visualisation, normalisation, and decision-making.

The turbines are close enough to share the same weather, so changes in windiness, gusts, and turbulence affect all three together.

The case study compares three turbines: WT-01, WT-02, WT-03

Before starting, run the mock telemetry generation script so that `messy_turbine_telemetry.csv` is in the same folder as this notebook.

If you are new to pandas or Plotly, open `snippets.ipynb` in this folder and copy the smallest pattern that matches the task.

## How to use this workbook

Read each section first, then fill the code cell below it. If a step is unfamiliar, use the matching example in `snippets.ipynb` and adapt it to the turbine columns. If you already know the pattern, write your own version and move on.

# Wind Turbine Telemetry Schema

## Dataset Grain

- One row per turbine per 10-minute timestamp.
- Three turbines are simulated: `WT-01`, `WT-02`, and `WT-03`.

## Column Dictionary

| Column | Type | Description | How it is used |
|---|---|---|---|
| `Timestamp` | datetime | 10-minute observation time for the record. | Used for time-series plots, trend fitting, rolling averages, and failure timing. |
| `Turbine_ID` | categorical | Turbine identifier, typically `WT-01`, `WT-02`, or `WT-03`. | Used to separate the turbine data |
| `Ambient_Temperature_C` | float | Local ambient temperature around the turbine nacelle area. | Environmental Variable |
| `Macro_Wind_Speed_mps` | float | Shared regional wind speed before local wake and gust effects. | The broader wind environment. |
| `Wind_Speed_mps` | float | Working wind speed experienced by the turbine after local effects. | Environmental Variable |
| `Wind_Gust_mps` | float | Short-term gust component added to the wind field. | Used to explore links with RPM and transient loading. |
| `Local_Wake_Turbulence_mps` | float | Local wake-induced turbulence component around each turbine. | Helps explain turbine-to-turbine differences in effective wind loading. |
| `Effective_Wind_Speed_mps` | float | Combined wind speed after macro wind, gust, and local wake effects. | Operating wind variable. |
| `Turbulence_Intensity` | float | Dimensionless turbulence metric derived from wind and gust variability. | Operating wind variable |
| `Atmospheric_Pressure_hPa` | float | Local atmospheric pressure. | Weather context variable |
| `Relative_Humidity_pct` | float | Relative humidity percentage. | Weather context variable |
| `Wind_Direction_deg` | float | Wind direction in degrees from 0 to 360. | Used in yaw error plots and directiona
| `Bearing_Temperature_C` | float | Bearing temperature in degrees Celsius. | Bearing health signal. |
| `Vibration_mm_s` | float | Vibration magnitude in mm/s. | Key condition-monitoring signal |
| `RPM` | float | Rotor rotational speed. | Sensor variable. |
| `Power_kW` | float | Electrical power output in kilowatts. | Sensor variable. |
| `Turbine_Load_Percent` | float | Power output scaled as a percentage of rated power. | Normalised load view for interpretation and comparison. |
| `Yaw_Error_deg` | float | Difference between wind direction and turbine yaw alignment. | Used in yaw misalignment exploration. |
| `Blade_Pitch_Deg` | float | Blade pitch angle in degrees. | Used in pitch-vs-power relationships and curtailment checks. |
| `Generator_Current_A` | float | Generator current in amps. | Electrical response variable that follows output changes. |
| `Nacelle_Temperature_C` | float | Nacelle temperature in degrees Celsius. | Secondary thermal indicator. |
| `Oil_Pressure_bar` | float | Hydraulic or lubrication oil pressure in bar. | Mechanical health indicator for drivetrain condition. |
| `Gearbox_Temperature_C` | float | Gearbox temperature in degrees Celsius. | Secondary drivetrain thermal health indicator. |
| `Operational_State` | categorical | Simulated operating mode such as `healthy`, `degrading`, `drifting`, or `offline`. | Useful for labels, filtering, and interpretation. |
| `Maintenance_Flag` | categorical | Human-readable maintenance condition such as `none`, `bearing degradation`, `catastrophic seizure`, or `early drift`. | Connects telemetry patterns to maintenance meaning. |
| `Grid_Priority` | categorical | Simplified grid dispatch label. | Included as a contextual field in exploratory charts. |

## Derived Columns Used in the Notebooks

These are created during analysis rather than written by the generator.

| Column | Type | Description |
|---|---|---|
| `Failure_State` | categorical | Notebook-created label used to mark rows with missing bearing temperature or vibration as sensor dropout / failure. |
| `Vibration_per_RPM` | float | Vibration normalised by RPM, used to compare roughness at different operating speeds. |
| `Vibration_per_RPM_Rolling` | float | Rolling mean of vibration per RPM for trend inspection. |
| `Day_Index` | float | Day offset from the start of the time series, used for linear trend fitting. |
| `Trend_Fit` | float | Fitted line for the WT-03 vibration-per-RPM trend. |


## Section 1: Data Ingestion & Data Cleaning [15 mins]

Start with the raw telemetry and decide what should be kept, cleaned, or set aside. In practice, missing values, flatlines, and sudden shifts in the new weather-related sensors are often part of the story rather than simple noise.

A row with missing values might mean a failed sensor, a broken data path, or a machine that has stopped. If we remove those rows too early, we lose useful evidence.

Keep the raw records, then create a cleaned analysis set for the charts and summary statistics.

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Load the telemetry CSV into df and inspect the data before changing anything.
# Use df.head(), df.info(), and df.isna().sum() so you can see the shape, column types, and missing values.
# Separate the shared weather columns from the turbine-specific sensor readings.
# Example starting point:
# df = pd.read_csv('messy_turbine_telemetry.csv', parse_dates=['Timestamp'])
# print(df.head())
raise NotImplementedError('Participant task: ingest the telemetry CSV and inspect it.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Keep the failure records in the raw data, but create a clean analysis copy for the charts.
# Missing values in key sensor columns can be evidence of failure — preserve them rather than filling.
# Sort by Turbine_ID and Timestamp before any rolling work.
# Example starting point:
# analysis_df = df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s']).copy()
# analysis_df = analysis_df.sort_values(['Turbine_ID', 'Timestamp'])
raise NotImplementedError('Participant task: clean the telemetry data with governance in mind.')

## Section 2: Interactive Anomaly Hunting via Scatter Plots [15 mins]

Temperature, vibration, RPM, and the new wind context fields often move together in ways that say something about asset health. Interactive plots make it easier to inspect the relationships and spot when a turbine starts to drift away from normal behaviour.

Use hover and zoom to inspect the point where behaviour changes.

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Build a Plotly scatter plot with vibration on the y-axis, temperature on the x-axis, turbine colour, and RPM as point size.
# Add hover details so the chart works well for anomaly spotting.
# Include wind speed, gusts, and turbulence in the hover so the weather context is visible.
# Example starting point:
# fig = px.scatter(
#     clean_df,
#     x='Bearing_Temperature_C',
#     y='Vibration_mm_s',
#     color='Turbine_ID',
#     size='RPM',
#     hover_data=['Wind_Speed_mps', 'Wind_Gust_mps', 'Turbulence_Intensity'],
# )
raise NotImplementedError('Participant task: build the multi-variable scatter plot.')

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Plot bearing temperature over time and keep the gaps visible where telemetry drops out.
# Make it easy to compare the three turbines and relate spikes to windy periods.
# Example starting point:
# fig = px.line(clean_df, x='Timestamp', y='Bearing_Temperature_C', color='Turbine_ID')
# fig.update_traces(connectgaps=False)
raise NotImplementedError('Participant task: build the time-series chart.')

### Interactive Challenge

Use Plotly's zoom and hover tools to find when WT-02 stops producing useful telemetry and when WT-03 starts to move away from the healthy pattern. The point is to connect the chart back to the asset, not just describe its shape.

## Section 3: Vibration Normalised by RPM [15 mins]

Rather than rescaling every feature separately, compare vibration against RPM. This gives a simple view of how much vibration the turbine is producing for a given operating speed and leaves room to judge whether the change is tied to weather or to asset condition.

A rising vibration-to-RPM ratio can be a useful warning sign when a machine starts to behave less smoothly, especially if the weather is not explaining the change.

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Calculate a vibration-to-RPM ratio for WT-03 using a zero-safe denominator.
# Plot the ratio over time and use it to support the analysis.
# Add wind speed or turbulence alongside the ratio if it helps explain the pattern.
# Example starting point:
# wt03 = clean_df.loc[clean_df['Turbine_ID'] == 'WT-03'].copy()
# wt03['Vibration_per_RPM'] = wt03['Vibration_mm_s'] / wt03['RPM'].replace(0, np.nan)
raise NotImplementedError('Participant task: compare WT-03 vibration against RPM.')

## Section 4: The Operational Decision & Dashboard Ethics [15 mins]

Dashboards are not neutral. They influence maintenance timing, budget decisions, and safety risk. Use the chart to show which turbine should be prioritised if only one overhaul can be funded this quarter.

Work out which one is healthy, which one has failed, and which one is likely to fail soon. Use the weather context and the sensor trends to justify the choice.

In [ ]:
# --- PARTICIPANT CODE HERE ---
# Summarise the remaining lifespan or maintenance priority so the choice is easy to explain.
# Use the latest weather and sensor readings as supporting evidence in the chart or notes.
# A short table plus one chart is enough if it makes the decision clear.
# Example starting point:
# summary = clean_df.groupby('Turbine_ID').agg({
#     'Power_kW': 'mean',
#     'Vibration_mm_s': 'max',
#     'Maintenance_Flag': 'last',
# })
raise NotImplementedError('Participant task: build the executive dashboard.')

### Ethics & Governance Challenge

Consider the full cost of the decision. Do you spend the budget replacing WT-02, which is already offline and represents lost revenue, or do you intervene on WT-03 before it fails in the same way?

Your answer should cover safety, environmental impact, operations, and liability. Data quality, visualisation design, and maintenance policy all feed into that choice.

## Wrap-Up

By the end of this workbook, you should be able to move from raw telemetry to a defensible operational recommendation using cleaning, plotting, normalisation, and a simple decision dashboard.

If you needed the starter patterns, check `snippets.ipynb` and then come back to this notebook to finish the analysis in your own words.